# Blend weights explorer

Interactively explore the blend weights produced by `calculate_hierarchical_blend` as the delay
of each model changes.

- Each model's slider sets how delayed its latest forecast is, in hours
- Tick a model's **exclude** box to leave it out of the blend
- Hover over the plot to see every model's weight at that horizon. Zoom by dragging a box, and
  double-click to reset. The zoom is kept as you move the controls

## Setting up the environment

This notebook needs `ipykernel`, `ipywidgets`, `plotly`, `anywidget` (which plotly's interactive
figures need) and `matplotlib`, none of which are project dependencies. From the repository root:

```bash
uv sync
uv pip install --python .venv ipykernel ipywidgets plotly anywidget matplotlib
```

- `uv sync` creates `.venv` (with Python 3.12, downloading it if needed), installs the locked
  dependencies, and installs `forecast_blend` in editable mode so code changes are picked up
- The second command adds the packages needed to run this notebook
- If `.venv` already exists, you only need the second command

Then select `.venv/bin/python` as the notebook kernel.

Note that a later `uv sync` removes packages that are not in the lockfile, so you will need to
re-run the install command afterwards.


In [50]:
import ipywidgets as widgets
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from IPython.display import display
from plotly.colors import qualitative

from forecast_blend.weights import calculate_hierarchical_blend, get_horizon_maes

In [51]:
df_mae = get_horizon_maes()

# Sort the models by longest forecast horizon, then by lowest mean MAE. This sets the order of the
# controls and the colour of each model
max_horizons = {model: df_mae[model].dropna().index.max() for model in df_mae.columns}
sorted_models = sorted(df_mae.columns, key=lambda m: (max_horizons[m], df_mae[m].mean()))
df_mae = df_mae[sorted_models]

df_mae

,pvnet_cloud,pvnet_v2,pvnet_day_ahead,pvnet_ecmwf,pvnet_ukv_only
horizon,,,,,
0 days 00:30:00,0.005589,0.005081,0.009917,0.010821,0.011252
0 days 01:00:00,0.005263,0.005300,0.009607,0.010418,0.011060
0 days 01:30:00,0.005561,0.005781,0.009538,0.010346,0.011048
0 days 02:00:00,0.005932,0.006372,0.009497,0.010229,0.011028
0 days 02:30:00,0.006331,0.006956,0.009554,0.010151,0.010991
...,...,...,...,...,...
1 days 10:00:00,NaN,NaN,0.013559,0.013348,0.020708
1 days 10:30:00,NaN,NaN,0.013626,0.013398,0.021983
1 days 11:00:00,NaN,NaN,0.013801,0.013415,0.023193


In [ ]:
x = df_mae.index.values.astype("timedelta64[s]").astype(float) / 3600

for col in df_mae.columns:
    y = df_mae[col]
    plt.plot(x, y, label=col)

plt.xlabel("Forecast horizon (hours)")
plt.ylabel("NMAE")
plt.legend()
plt.show()

In [ ]:
MAX_HORIZON_HOURS = df_mae.index.max() / pd.Timedelta("1h")


# Each model keeps the same colour however the controls change
MODEL_COLOURS = dict(zip(df_mae.columns, qualitative.Plotly))


def make_model_row(model: str) -> tuple[widgets.HBox, widgets.FloatSlider, widgets.Checkbox]:
    """A row of controls for one model: its delay slider and an exclude box"""
    delay = widgets.FloatSlider(
        value=0,
        min=0,
        max=MAX_HORIZON_HOURS,
        step=0.5,
        description=model,
        readout_format=".1f",
        continuous_update=False,  # Only recompute when the slider is released
        style={"description_width": "130px"},  # Same label width so all sliders line up
        layout=widgets.Layout(width="650px"),
        tooltip=f"Hours since the latest {model} forecast was initialised",
    )
    exclude = widgets.Checkbox(value=False, description="exclude from blend", indent=False)
    row = widgets.HBox([delay, widgets.Label("h", layout=widgets.Layout(width="30px")), exclude])
    return row, delay, exclude


rows, delay_sliders, exclude_boxes = {}, {}, {}
for model in df_mae.columns:
    rows[model], delay_sliders[model], exclude_boxes[model] = make_model_row(model)

header = widgets.HTML("<b>Delay of each model's latest forecast</b>")
status = widgets.HTML()

# The figure is displayed once and its traces are updated in place when the controls change.
# There is one trace per model, hidden when the model is not in the blend
fig = go.FigureWidget(
    layout=dict(
        xaxis=dict(
            title="Forecast horizon (hours)",
            range=[0, MAX_HORIZON_HOURS],
            ticks="outside",
            # An unlabelled tick mark every hour between the labelled ticks
            minor=dict(dtick=1, ticks="outside", ticklen=4, showgrid=False),
        ),
        yaxis=dict(title="Blend weight", range=[-0.05, 1.05]),
        # Show every model's weight at the hovered horizon in a single tooltip
        hovermode="x unified",
        # Legend above the plot, so its width never changes the size of the plot area
        legend=dict(orientation="h", x=0, y=1.02, xanchor="left", yanchor="bottom"),
        # Keep the user's zoom, pan and legend selections when the data is updated
        uirevision="keep",
        height=480,
        width=900,
        margin=dict(l=60, r=20, t=60, b=50),
    )
)
for model in df_mae.columns:
    fig.add_scatter(
        name=model,
        mode="lines",
        line=dict(color=MODEL_COLOURS[model]),
        hovertemplate="%{y:.2f}",
    )


def update_plot(change: dict | None = None) -> None:
    # Round to the 30 minute MAE step. The slider can report values like 1.4999999999999998, and a
    # delay even a nanosecond off the step misaligns the models' horizons and breaks the blend
    model_delays_dict = {
        model: pd.Timedelta(hours=delay_sliders[model].value).round("30min")
        for model in df_mae.columns
        if not exclude_boxes[model].value
    }

    try:
        df_weights = calculate_hierarchical_blend(
            df_mae=df_mae[list(model_delays_dict)],
            model_delays_dict=model_delays_dict,
        )
    except ValueError as e:
        # E.g. every model is excluded or delayed past the end of its forecast horizon
        status.value = f"<span style='color: #c0392b'>No blend can be made: {e}</span>"
        with fig.batch_update():
            for trace in fig.data:
                trace.visible = False
        return

    status.value = ""
    df_weights = df_weights.fillna(0)

    # Plain lists rather than numpy arrays: FigureWidget fails when the frontend syncs numpy
    # arrays back to Python ("truth value of an array ... is ambiguous")
    x = (df_weights.index / pd.Timedelta("1h")).tolist()

    with fig.batch_update():
        for trace in fig.data:
            if trace.name in df_weights.columns:
                trace.x = x
                trace.y = df_weights[trace.name].tolist()
                trace.visible = True
            else:
                trace.visible = False


for control in [*delay_sliders.values(), *exclude_boxes.values()]:
    control.observe(update_plot, names="value")

update_plot()
display(widgets.VBox([header, *rows.values(), status, fig]))